In [0]:
%sql
SELECT * FROM gold.dim_date ORDER BY 1 ASC LIMIT 1;

In [0]:
%sql
SELECT MIN(commit_author_date) AS earliest_author_date,
       MIN(commit_committer_date) AS earliest_committer_date
FROM silver.runtime_commits;

In [0]:
%sql
SELECT r.name,
       SUM(CASE WHEN f.commit_author_date_key = -1 THEN 1 ELSE 0 END) AS missing_author_key
FROM gold.fact_commits f JOIN gold.dim_repo r ON f.repo_key = r.repo_key
GROUP BY r.name ORDER BY missing_author_key DESC;

In [0]:
%sql
SELECT CASE WHEN time_to_commit_hours < 0 THEN 'negative'
            WHEN time_to_commit_hours > 8760 THEN 'over 1 year'
            ELSE 'normal' END AS bucket,
       commit_author_date_key = -1 AS author_key_missing,
       COUNT(*) AS commits
FROM gold.fact_commits
GROUP BY 1, 2 ORDER BY 1, 2;

In [0]:
%sql
SELECT f.assignees_count, COUNT(*) AS prs
FROM gold.fact_pull_request f
JOIN silver.vscode_pr_assignees a ON f.pr_id = a.pr_id
WHERE a.assignee_id_is_missing = 1
GROUP BY f.assignees_count;

In [0]:
%sql
SELECT f.reviewers_count, COUNT(*) AS prs
FROM gold.fact_pull_request f
JOIN silver.vscode_pr_reviewers rv ON f.pr_id = rv.pr_id
WHERE rv.reviewer_id_is_missing = 1
GROUP BY f.reviewers_count;

In [0]:
%sql
SELECT
  (SELECT SUM(assignees_count) FROM gold.fact_pull_request f JOIN gold.dim_repo r ON f.repo_key = r.repo_key WHERE r.name = 'vscode') AS fact_assignees,
  (SELECT COUNT(*) FROM silver.vscode_pr_assignees WHERE assignee_id_is_missing = 0) AS real_assignees,
  (SELECT SUM(reviewers_count) FROM gold.fact_pull_request f JOIN gold.dim_repo r ON f.repo_key = r.repo_key WHERE r.name = 'vscode') AS fact_reviewers,
  (SELECT COUNT(*) FROM silver.vscode_pr_reviewers WHERE reviewer_id_is_missing = 0) AS real_reviewers;

In [0]:
%sql
DESCRIBE gold.fact_pull_request;

In [0]:
%sql
SELECT * FROM gold.dim_flags;

In [0]:
%sql
SELECT f.flag_key, COUNT(*) AS prs
FROM gold.fact_pull_request f
LEFT JOIN gold.dim_flags d ON f.flag_key = d.flag_key
WHERE d.flag_key IS NULL
GROUP BY f.flag_key;

In [0]:
%sql
SELECT user_key, user_id, user_login FROM gold.dim_user WHERE user_login = 'Copilot';

In [0]:
%sql
SELECT COUNT(*) AS prs_by_copilot FROM gold.fact_pull_request f
JOIN gold.dim_user u ON f.author_key = u.user_key WHERE u.user_login = 'Copilot';

In [0]:
%sql
SELECT
  (SELECT COUNT(DISTINCT pr_id) FROM silver.vscode_pull_requests) AS silver_prs,
  (SELECT COUNT(DISTINCT f.pr_id) FROM gold.fact_pull_request f JOIN gold.dim_repo r ON f.repo_key = r.repo_key WHERE r.name = 'vscode') AS gold_prs;

In [0]:
%sql
SELECT COUNT(DISTINCT pr_id) FROM gold.fact_pull_request;

In [0]:
%sql
SELECT MIN(s.commit_author_date) AS min_author_date,
       MAX(s.commit_author_date) AS max_author_date,
       COUNT(*) AS rows_checked
FROM gold.fact_commits f
JOIN (
  SELECT sha, commit_author_date FROM silver.vscode_commits UNION ALL
  SELECT sha, commit_author_date FROM silver.aspnetcore_commits UNION ALL
  SELECT sha, commit_author_date FROM silver.powershell_commits UNION ALL
  SELECT sha, commit_author_date FROM silver.powertoys_commits UNION ALL
  SELECT sha, commit_author_date FROM silver.runtime_commits UNION ALL
  SELECT sha, commit_author_date FROM silver.terminal_commits UNION ALL
  SELECT sha, commit_author_date FROM silver.typescript_commits
) s ON f.sha = s.sha
WHERE f.commit_author_date_key = -1;